# SwarmSort Gen 2, notebook 4: full training of the Arm B winner (random search)

**What:** fully trains the best configuration found by the random search search (`g2_3_search_random.ipynb`):
`results/gen2/best_random.json`. It uses exactly the Arm A recipe: 100 epochs, imgsz 640, batch 16, seed 42, deterministic, SGD, warmup 3
epochs, mosaic closed for the last 10, one GPU. All of this is fixed in `src/training.py`, so only the six searched
values differ from Arm A and the comparison is fair. The reported numbers are the val metrics of `best.pt`; the test
split is used only once, in `g2_6_evaluate`.

**Why:** the search ranked configurations with cheap 12-epoch proxies; only a full training shows what the winner is
worth.

**Before you run it: commit `results/gen2/best_random.json` first, or attach notebook 3's output.** The file is
written by `g2_3_search_random`. Either download it, commit and push it (as in Gen 1), or Add Input -> Notebook Output ->
`g2_3_search_random`; the fallback cell copies it from there if the repository lacks it.

**Inputs:** Add Input -> Notebook Output -> **`g2_0_build_dataset`** (the dataset), and **`g2_3_search_random`** if
`results/gen2/best_random.json` is not committed. **Accelerator: GPU T4 x2**, **Internet: on**. Run it with
**Save Version -> Save & Run All (Commit)**.

**Expected time:** about 2.35 hours (Gen 1's Arm B took 4157 s; 4157 s x 2.033 = 8,450 s = 2.35 h). Time estimates scale the measured Gen 1 times (GPU T4, imgsz 416, 7324 train images) by the pixel ratio (640/416)² = 2.367 and the train-image ratio 6290/7324 = 0.859, a factor of **2.033**. The expected 6290 Gen 2 train images come from the source research: Gen 1 subsample 3000 + 70% of TACO's 1500 (1050) + 70% of HITL's 3200 (2240). The real count is in `build_report.json`. The images are stored with a long side of up to 1280 px, so if JPEG decoding on Kaggle's 4 CPU cores limits the speed, allow up to about 30% more.

This notebook and `g2_5_arm_c` are independent: they can run at the same time in separate Kaggle sessions.

Outputs: `/kaggle/working/SwarmSort/results/gen2/arm_b.json`, `results/gen2/arm_b_epochs.csv` and the run folder
`results/gen2/runs/arm_b/`; the summary cell copies the weights to `/kaggle/working/gen2_arm_b_best.pt`.

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4 x2 and run again"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))


def run(*args):
    """Run a repository script. check=True: if the script stops with an error, this cell fails and the version
    is marked failed."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    subprocess.run([sys.executable, *args], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Find the dataset built by g2_0_build_dataset (attached as input) and write configs that point at it:
# /kaggle/working/gen2_cfg/data.yaml, data_test_<domain>.yaml, split.csv and lists/ (the images stay read-only)
import glob
INPUT, CFG = "/kaggle/input", "/kaggle/working/gen2_cfg"
builds = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/build_report.json", recursive=True)
                 if os.path.isfile(os.path.join(os.path.dirname(p), "data.yaml"))
                 and os.path.isdir(os.path.join(os.path.dirname(p), "images"))})
if not builds:
    raise FileNotFoundError("The Generation 2 dataset is not attached. Add Input -> Notebook Output -> "
                            "g2_0_build_dataset (its finished version), then run again.")
if len(builds) > 1:
    raise RuntimeError("Several built datasets are attached; remove all but one version of g2_0_build_dataset:\n  "
                       + "\n  ".join(builds))
BUILD = builds[0]
print("Built dataset:", BUILD)
if "--relocate-from" not in open("src/build_dataset.py", encoding="utf-8").read():
    raise RuntimeError("src/build_dataset.py on GitHub has no --relocate-from mode yet: push the latest commits")
run("src/build_dataset.py", "--relocate-from", BUILD, "--out", CFG)
assert os.path.isfile(f"{CFG}/data.yaml"), f"the relocation did not write {CFG}/data.yaml"
print(open(f"{CFG}/data.yaml").read())
print("Test-set yamls:", sorted(os.path.basename(p) for p in glob.glob(f"{CFG}/data_test_*.yaml")))


In [ ]:
# results/gen2/best_random.json: the committed file, or else the one in an attached g2_3_search_random output
import json, shutil
CONFIG, METHOD, SEARCH_NB = "results/gen2/best_random.json", "random", "g2_3_search_random"


def full_search(best):
    return best.get("method") == METHOD and not best.get("smoke") and best.get("imgsz") == 640 and "params" in best


if os.path.exists(CONFIG):
    print(CONFIG, "found in the repository")
else:
    found = {}
    for path in sorted(glob.glob(f"{INPUT}/**/{CONFIG}", recursive=True)):
        try:
            best = json.load(open(path))
        except (OSError, ValueError):
            print("Skipping unreadable", path)
            continue
        if not full_search(best):
            print(f"Skipping {path}: not a full {METHOD} search at imgsz 640")
            continue
        found.setdefault(json.dumps(best["params"], sort_keys=True), path)
    if not found:
        raise FileNotFoundError(f"{CONFIG} is not in the repository and no attached notebook output has it. Commit "
                                f"and push it first, or Add Input -> Notebook Output -> {SEARCH_NB} (its finished "
                                f"version), then run again.")
    if len(found) > 1:
        raise RuntimeError(f"The attached outputs hold different {CONFIG} files; keep only one version of "
                           f"{SEARCH_NB}:\n  " + "\n  ".join(found.values()))
    path = next(iter(found.values()))
    os.makedirs(os.path.dirname(CONFIG), exist_ok=True)
    shutil.copy(path, CONFIG)
    print(f"Copied {path} -> {CONFIG}")
best = json.load(open(CONFIG))
assert full_search(best), f"{CONFIG} is not the result of a full {METHOD} search at imgsz 640"
sys.path.insert(0, "src")
from search_space import DEFAULTS, NAMES
print(f"Arm B winner: {best['label']}, proxy val mAP@50 {best['map50']:.4f} ({best['proxy_epochs']}-epoch proxy)")
print(f"\n{'hyperparameter':<14}{'Arm B (trained here)':>24}{'Arm A (defaults)':>18}")
for k in NAMES:
    print(f"{k:<14}{best['params'][k]:>24g}{DEFAULTS[k]:>18g}")


In [ ]:
# Full training of the Arm B winner on one GPU (device 0), exactly as Arm A was trained
run("src/train_final.py", "--config", CONFIG, "--data", f"{CFG}/data.yaml", "--imgsz", 640, "--name", "arm_b",
    "--out", "results/gen2/arm_b.json", "--project", "results/gen2/runs", "--device", 0)


In [ ]:
# Arm B next to Arm A (val split, best.pt) if Arm A's JSON is available, a check that the intended config was
# trained, and a copy of the weights at the top of /kaggle/working so they are easy to find in the Output tab
import math
ARM, LETTER = "arm_b", "B"
res = json.load(open(f"results/gen2/{ARM}.json"))
wanted = json.load(open(CONFIG))["params"]
wrong = [k for k in NAMES if not math.isclose(res["params"][k], wanted[k], rel_tol=1e-9)]
assert not wrong, f"results/gen2/{ARM}.json was trained with other values than {CONFIG} for {wrong}"
assert res["epochs"] == 100 and res["imgsz"] == 640 and not res["smoke"], "not the full 100-epoch recipe at imgsz 640"
print(f"Trained config matches {CONFIG}:", ", ".join(f"{k}={res['params'][k]:g}" for k in NAMES))

# Arm A: committed results/gen2/arm_a.json, or one in an attached g2_1_arm_a output (optional)
candidates = ["results/gen2/arm_a.json"] + sorted(glob.glob(f"{INPUT}/**/results/gen2/arm_a.json", recursive=True))
base_path = next((p for p in candidates if os.path.exists(p)), None)
if base_path:
    base = json.load(open(base_path))
    print(f"\n{'val split, best.pt':<20}{'Arm ' + LETTER:>12}{'Arm A':>12}{'difference':>12}   (Arm A: {base_path})")
    for key, label in [("map50", "mAP@50"), ("map50_95", "mAP@50-95"), ("precision", "precision"), ("recall", "recall")]:
        print(f"{label:<20}{res[key]:>12.4f}{base[key]:>12.4f}{res[key] - base[key]:>+12.4f}")
    print(f"{'training time (h)':<20}{res['seconds'] / 3600:>12.2f}{base['seconds'] / 3600:>12.2f}")
else:
    print(f"\nval split, best.pt (Arm A's JSON not available for a side-by-side view)")
    for key, label in [("map50", "mAP@50"), ("map50_95", "mAP@50-95"), ("precision", "precision"), ("recall", "recall")]:
        print(f"  {label:<12}{res[key]:.4f}")
    print(f"  training time {res['seconds'] / 3600:.2f} h")
print(f"GPU: {res['device']}")
print("Val numbers only: the fair comparison is the test split, evaluated once in g2_6_evaluate.")

weights = f"results/gen2/runs/{ARM}/weights/best.pt"
assert os.path.exists(weights), f"{weights} not found (see the output of the training cell)"
shutil.copyfile(weights, f"/kaggle/working/gen2_{ARM}_best.pt")
print(f"\nCopied {weights} -> /kaggle/working/gen2_{ARM}_best.pt ({os.path.getsize(weights) / 1e6:.1f} MB)")


### Files to download / where they go locally

| File in the Output tab | Where it goes locally |
|---|---|
| `SwarmSort/results/gen2/arm_b.json` | `results/gen2/arm_b.json` |
| `SwarmSort/results/gen2/arm_b_epochs.csv` | `results/gen2/arm_b_epochs.csv` |
| `gen2_arm_b_best.pt` (top level of the output) | `results/gen2/runs/arm_b/weights/best.pt` (create the folders) |

**The `.pt` file downloads as a `.zip`**, because a PyTorch checkpoint is itself a zip archive. Do not unzip it: just
rename the downloaded file to `best.pt`. Commit the JSON and CSV; the weights stay local (git-ignored).
`g2_6_evaluate` takes the weights straight from this notebook's output, so attach this version there.

### If the session dies

This is one training (about 2.35 hours), well inside Kaggle's 12-hour limit, so a dead session is simply rerun with
the same settings: the run is seeded and deterministic. Nothing needs to be resumed.